In [ ]:
%matplotlib widget

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import (
    FloatSlider,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)
from IPython.display import display, Math

# ============================================================
# LAPLACE EQUATION IN THE UPPER HALF-PLANE
#
# phi_xx + phi_yy = 0
#
# y > 0
#
# phi(x,0) = f(x)
#
# f(x) = 1,  |x| < 1
#        0,  elsewhere
#
# The problem-specific results below are calculated
# symbolically by SymPy.
# ============================================================

plt.ioff()

# ============================================================
# DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea,
.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.lap-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.lap-label {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.lap-value {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1150px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.50;
    margin-bottom:10px;
">

<div class="lap-title" style="margin-bottom:8px;">
Symbolic Solution of Laplace's Equation in the Upper Half-Plane
</div>

<div style="margin-bottom:5px;">
The equation φₓₓ+φᵧᵧ=0 is solved for y&gt;0 with boundary
condition φ(x,0)=f(x) and with φ bounded as y→∞.
</div>

<div style="margin-bottom:5px;">
The Fourier transform is taken with respect to x. The transformed
partial differential equation becomes an ordinary differential
equation in y.
</div>

<div style="margin-bottom:5px;">
For a completely symbolic demonstration, the boundary function is
chosen as the rectangular pulse f(x)=1 for |x|&lt;1 and f(x)=0
elsewhere.
</div>

<div>
<b>All problem-specific quantities</b> — the Fourier transform of the
boundary data, the transformed solution, the inverse transform of the
exponential factor, the Poisson kernel, and the final harmonic solution —
are calculated symbolically by SymPy.
</div>

</div>
""")

display(documentation)

# ============================================================
# SYMBOLS
# ============================================================

x = sp.symbols('x', real=True)
xi = sp.symbols('xi', real=True)
omega = sp.symbols('omega', real=True)

# y is strictly positive because the problem is defined
# in the upper half-plane.
y = sp.symbols('y', positive=True, real=True)

I = sp.I

Phi = sp.Function('Phi')

# ============================================================
# HELPER FOR PIECEWISE RESULTS
# ============================================================

def first_piece(expr):
    if isinstance(expr, sp.Piecewise):
        return sp.simplify(expr.args[0][0])
    return sp.simplify(expr)

# ============================================================
# ORIGINAL PDE
# ============================================================

phi_symbolic = sp.Function('phi')

original_pde = sp.Eq(
    sp.diff(phi_symbolic(x, y), x, 2)
    +
    sp.diff(phi_symbolic(x, y), y, 2),
    0
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:8px;
">
Original boundary-value problem
</div>
"""))

display(Math(sp.latex(original_pde)))

display(
    Math(
        r'\varphi(x,0)=f(x)'
    )
)

# ============================================================
# GENERAL FOURIER DIFFERENTIATION RULES
#
# These are theoretical properties of the transform,
# not results specific to the exercise.
# ============================================================

Phi_expr = Phi(omega, y)

transform_xx = sp.simplify(
    (I * omega)**2
    *
    Phi_expr
)

transform_yy = sp.diff(
    Phi_expr,
    y,
    2
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Fourier transformation with respect to x
</div>
"""))

display(
    Math(
        r'\mathcal{F}_x\{\varphi_{xx}\}'
        r'='
        +
        sp.latex(transform_xx)
    )
)

display(
    Math(
        r'\mathcal{F}_x\{\varphi_{yy}\}'
        r'='
        +
        sp.latex(transform_yy)
    )
)

# ============================================================
# TRANSFORMED ODE
# ============================================================

transformed_ode = sp.Eq(
    sp.simplify(
        transform_xx
        +
        transform_yy
    ),
    0
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Transformed differential equation
</div>
"""))

display(
    Math(
        sp.latex(transformed_ode)
    )
)

# ============================================================
# SOLVE THE y-DEPENDENCE SYMBOLICALLY
#
# Introduce k > 0 to represent |omega|.
# ============================================================

k = sp.symbols(
    'k',
    positive=True,
    real=True
)

r = sp.symbols('r')

characteristic_equation = sp.Eq(
    r**2 - k**2,
    0
)

characteristic_roots = sp.solve(
    characteristic_equation,
    r
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Characteristic equation in y
</div>
"""))

display(
    Math(
        sp.latex(
            characteristic_equation
        )
    )
)

display(
    Math(
        r'r='
        +
        sp.latex(
            characteristic_roots
        )
    )
)

# ============================================================
# SELECT THE BOUNDED ROOT AUTOMATICALLY
# ============================================================

bounded_roots = [
    root
    for root in characteristic_roots
    if root.is_negative
]

bounded_root = bounded_roots[0]

display(
    Math(
        r'r_{\mathrm{bounded}}='
        +
        sp.latex(
            bounded_root
        )
    )
)

# ============================================================
# GENERAL BOUNDED TRANSFORMED SOLUTION
# ============================================================

B = sp.symbols('B')

Phi_bounded_k = (
    B
    *
    sp.exp(
        bounded_root * y
    )
)

display(
    Math(
        r'\Phi(\omega,y)='
        +
        sp.latex(
            Phi_bounded_k
        )
    )
)

# ============================================================
# BOUNDARY FUNCTION
#
# f(x) = 1 for -1 < x < 1
# ============================================================

f_boundary = sp.Piecewise(
    (1, sp.Abs(x) < 1),
    (0, True)
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Boundary function
</div>
"""))

display(
    Math(
        r'f(x)='
        +
        sp.latex(
            f_boundary
        )
    )
)

# ============================================================
# SYMBOLIC FOURIER TRANSFORM OF f(x)
#
# Since f(x)=0 outside [-1,1],
#
# F(omega) = integral_-1^1 exp(-i omega x) dx
# ============================================================

F_raw = sp.integrate(
    sp.exp(
        -I * omega * xi
    ),
    (
        xi,
        -1,
        1
    )
)

F_boundary = first_piece(
    F_raw
)

F_boundary = sp.trigsimp(
    sp.simplify(
        F_boundary
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Symbolic Fourier transform of the boundary data
</div>
"""))

display(
    Math(
        r'F(\omega)='
        r'\int_{-1}^{1}'
        r'e^{-i\omega x}\,dx'
    )
)

display(
    Math(
        r'F(\omega)='
        +
        sp.latex(
            F_boundary
        )
    )
)

# ============================================================
# APPLY THE TRANSFORMED BOUNDARY CONDITION
#
# Phi(omega,0) = F(omega)
# ============================================================

B_solution = sp.solve(
    sp.Eq(
        Phi_bounded_k.subs(
            y,
            0
        ),
        F_boundary
    ),
    B
)[0]

Phi_k = sp.simplify(
    Phi_bounded_k.subs(
        B,
        B_solution
    )
)

# Replace k by |omega|.
Phi_final = sp.simplify(
    Phi_k.subs(
        k,
        sp.Abs(omega)
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Transformed solution
</div>
"""))

display(
    Math(
        r'\Phi(\omega,y)='
        +
        sp.latex(
            Phi_final
        )
    )
)

# ============================================================
# SYMBOLIC DERIVATION OF THE POISSON KERNEL
#
# Compute:
#
# g(x,y) =
# F^{-1}{exp(-|omega| y)}
#
# by splitting the inverse-transform integral
# at omega = 0.
# ============================================================

negative_kernel_raw = sp.integrate(
    sp.exp(
        omega * y
    )
    *
    sp.exp(
        I * omega * x
    ),
    (
        omega,
        -sp.oo,
        0
    )
)

positive_kernel_raw = sp.integrate(
    sp.exp(
        -omega * y
    )
    *
    sp.exp(
        I * omega * x
    ),
    (
        omega,
        0,
        sp.oo
    )
)

negative_kernel = first_piece(
    negative_kernel_raw
)

positive_kernel = first_piece(
    positive_kernel_raw
)

poisson_kernel = sp.simplify(
    (
        negative_kernel
        +
        positive_kernel
    )
    /
    (
        2 * sp.pi
    )
)

poisson_kernel = sp.factor(
    poisson_kernel
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Symbolic inverse transform of the exponential factor
</div>
"""))

display(
    Math(
        r'g(x,y)='
        r'\mathcal{F}^{-1}'
        r'\left\{e^{-|\omega|y}\right\}'
    )
)

display(
    Math(
        r'g(x,y)='
        +
        sp.latex(
            poisson_kernel
        )
    )
)

# ============================================================
# SYMBOLIC CONVOLUTION WITH THE RECTANGULAR BOUNDARY DATA
#
# Because f(xi)=1 only for -1 < xi < 1:
#
# phi(x,y) = integral_-1^1 g(x-xi,y) dxi
# ============================================================

kernel_shifted = sp.simplify(
    poisson_kernel.subs(
        x,
        x - xi
    )
)

phi_solution_raw = sp.integrate(
    kernel_shifted,
    (
        xi,
        -1,
        1
    )
)

phi_solution = sp.simplify(
    phi_solution_raw
)

phi_solution = sp.trigsimp(
    phi_solution
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:19px;
    font-weight:bold;
    color:#16802b;
    margin-top:18px;
">
Final symbolic solution
</div>
"""))

display(
    Math(
        r'\varphi(x,y)='
        +
        sp.latex(
            phi_solution
        )
    )
)

# ============================================================
# SYMBOLIC VERIFICATION OF LAPLACE EQUATION
# ============================================================

laplace_residual = sp.simplify(
    sp.diff(
        phi_solution,
        x,
        2
    )
    +
    sp.diff(
        phi_solution,
        y,
        2
    )
)

display(HTML("""
<div style="
    font-family:Arial;
    font-size:17px;
    font-weight:bold;
    color:#6f3fa0;
    margin-top:14px;
">
Symbolic verification
</div>
"""))

display(
    Math(
        r'\varphi_{xx}+\varphi_{yy}'
        r'='
        +
        sp.latex(
            laplace_residual
        )
    )
)

# ============================================================
# NUMERICAL FUNCTION GENERATED FROM THE SYMBOLIC SOLUTION
# ============================================================

phi_numeric = sp.lambdify(
    (x, y),
    phi_solution,
    modules='numpy'
)

# ============================================================
# NUMERICAL GRID FOR VISUALIZATION
# ============================================================

XMAX = 6.0
NX = 700

x_values = np.linspace(
    -XMAX,
    XMAX,
    NX
)

YMAX = 3.0
NY = 120

y_values = np.linspace(
    0.05,
    YMAX,
    NY
)

X_grid, Y_grid = np.meshgrid(
    x_values,
    y_values
)

field_values = phi_numeric(
    X_grid,
    Y_grid
)

# ============================================================
# BOUNDARY DATA FOR THE GRAPH ONLY
# ============================================================

boundary_values = (
    np.abs(
        x_values
    )
    <
    1.0
).astype(float)

# ============================================================
# SLIDER
# ============================================================

y_slider = FloatSlider(
    min=0.05,
    max=YMAX,
    step=0.05,
    value=0.50,
    readout=False,
    continuous_update=True,
    layout=Layout(
        width='260px'
    )
)

y_value = HTML(
    '<div class="lap-value">0.50</div>'
)

height_row = HBox(
    [
        HTML(
            '<div class="lap-label">Height y:</div>',
            layout=Layout(
                width='100px',
                min_width='100px'
            )
        ),
        y_slider,
        y_value
    ],
    layout=Layout(
        width='450px',
        height='40px',
        align_items='center'
    )
)

controls = VBox(
    [
        HTML(
            '<div class="lap-title">Visualization</div>'
        ),
        height_row
    ],
    layout=Layout(
        width='500px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

current_math = HTMLMath()

current_panel = VBox(
    [
        HTML(
            '<div class="lap-title">Current Slice</div>'
        ),
        current_math
    ],
    layout=Layout(
        width='600px',
        padding='10px 14px',
        border='1px solid #d2c2df'
    )
)

top_row = HBox(
    [
        controls,
        current_panel
    ],
    layout=Layout(
        width='1120px',
        gap='15px',
        align_items='stretch'
    )
)

# ============================================================
# INITIAL SLICE
# ============================================================

initial_y = y_slider.value

initial_slice = phi_numeric(
    x_values,
    initial_y
)

# ============================================================
# FIGURE 1
# ============================================================

fig_slice, ax_slice = plt.subplots(
    figsize=(6.3, 4.8)
)

fig_slice.canvas.header_visible = False
fig_slice.canvas.footer_visible = False
fig_slice.canvas.toolbar_visible = False

fig_slice.canvas.layout = Layout(
    width='630px',
    height='480px'
)

ax_slice.set_title(
    'Boundary Data and Harmonic Extension',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_slice.set_xlabel('x')
ax_slice.set_ylabel('Value')

ax_slice.set_xlim(
    -XMAX,
    XMAX
)

ax_slice.set_ylim(
    -0.10,
    1.20
)

ax_slice.grid(
    True,
    linestyle=':',
    alpha=0.40
)

boundary_line, = ax_slice.plot(
    x_values,
    boundary_values,
    linewidth=1.6,
    linestyle='--',
    label='f(x)'
)

solution_line, = ax_slice.plot(
    x_values,
    initial_slice,
    linewidth=2.2,
    label='φ(x,y)'
)

ax_slice.legend(
    loc='upper right',
    fontsize=9
)

fig_slice.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.13
)

# ============================================================
# FIGURE 2
# ============================================================

fig_field, ax_field = plt.subplots(
    figsize=(5.0, 4.8)
)

fig_field.canvas.header_visible = False
fig_field.canvas.footer_visible = False
fig_field.canvas.toolbar_visible = False

fig_field.canvas.layout = Layout(
    width='500px',
    height='480px'
)

ax_field.set_title(
    'Harmonic Solution in the Upper Half-Plane',
    fontsize=12,
    fontweight='bold',
    color='#0b3d91'
)

ax_field.set_xlabel('x')
ax_field.set_ylabel('y')

field_image = ax_field.imshow(
    field_values,
    extent=[
        -XMAX,
        XMAX,
        y_values[0],
        y_values[-1]
    ],
    origin='lower',
    aspect='auto',
    vmin=0.0,
    vmax=1.0,
    interpolation='bilinear'
)

selected_height_line = ax_field.axhline(
    initial_y,
    linestyle='--',
    linewidth=1.3
)

fig_field.colorbar(
    field_image,
    ax=ax_field,
    fraction=0.046,
    pad=0.04
)

fig_field.subplots_adjust(
    left=0.13,
    right=0.90,
    top=0.90,
    bottom=0.13
)

figures_row = HBox(
    [
        fig_slice.canvas,
        fig_field.canvas
    ],
    layout=Layout(
        width='1140px',
        gap='10px',
        align_items='flex-start'
    )
)

# ============================================================
# UPDATE
# ============================================================

def update_height(change=None):

    y_current = y_slider.value

    solution_values = phi_numeric(
        x_values,
        y_current
    )

    solution_line.set_ydata(
        solution_values
    )

    selected_height_line.set_ydata(
        [
            y_current,
            y_current
        ]
    )

    y_value.value = (
        f'<div class="lap-value">{y_current:.2f}</div>'
    )

    current_math.value = (
        r'\('
        r'y='
        +
        f'{y_current:.4f}'
        +
        r'\)'
    )

    fig_slice.canvas.draw_idle()
    fig_field.canvas.draw_idle()

y_slider.observe(
    update_height,
    names='value'
)

update_height()

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    width:1120px;
    padding:10px 13px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.55;
    box-sizing:border-box;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Interpretation
</div>

<div style="margin-bottom:6px;">
The Fourier transformation with respect to x converts Laplace's
partial differential equation into an ordinary differential equation
in y. The boundedness requirement automatically selects the
exponentially decaying solution in the transformed domain.
</div>

<div style="margin-bottom:6px;">
The Fourier transform of the rectangular boundary data is calculated
directly from its defining integral. The inverse transform of the
factor exp(−|ω|y) is also evaluated symbolically and produces the
Poisson kernel.
</div>

<div style="margin-bottom:6px;">
The final harmonic solution is obtained symbolically by convolving
this kernel with the boundary function. No final Poisson formula is
entered manually.
</div>

<div>
The plots are numerical evaluations of the symbolic expression found
above. Increasing y makes the boundary discontinuities progressively
smoother, illustrating the averaging property of harmonic extension.
</div>

</div>
""")

# ============================================================
# DISPLAY INTERACTIVE PART
# ============================================================

display(
    VBox(
        [
            top_row,
            figures_row,
            interpretation
        ],
        layout=Layout(
            width='1160px',
            gap='10px'
        )
    )
)